# 02 — Homography: camera image → Room 204 floor map

Step A6. Saves each camera's first frame, holds the (empty) calibration points, computes
`H` with `cv2.findHomography` once four point pairs are filled in, and maps a track box's feet onto the map.

The recordings have no floor markers, so the point pairs for cam_1 and cam_2 come from `ground_model.py`:
a pinhole model whose height and horizon are fitted from the tracked people (1.7 m tall) and whose map
position is fixed by the classroom door (`data/calibration/room204_map.json`). It is an estimate.
**To calibrate properly:** put four tape crosses on the floor, measure them on the map, and replace
the pairs in `CALIB` with those.

In [1]:
import json, sys
from pathlib import Path

import cv2
import numpy as np

REPO = Path.cwd().resolve()
while not (REPO / "data" / "video_config.json").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml" / "tracking"))
from paths import camera_ids, configured_take, video_path

TAKE = configured_take()
CAMS = camera_ids()
CALIB_DIR = REPO / "data" / "calibration"
CALIB_DIR.mkdir(parents=True, exist_ok=True)
print("take", TAKE, "| cameras", CAMS)

take 3 | cameras ['cam_1', 'cam_2', 'attendance_cam']


## 1. First frame of each camera

In [2]:
FRAME_SIZE = {}  # cam_id -> (width, height) in pixels
for cam in CAMS:
    src = video_path(cam, TAKE)
    cap = cv2.VideoCapture(str(src))
    ok, frame = cap.read()
    cap.release()
    assert ok, f"could not read the first frame of {src.name}"
    out = CALIB_DIR / f"{cam}_frame.png"
    cv2.imwrite(str(out), frame)
    h, w = frame.shape[:2]
    FRAME_SIZE[cam] = (w, h)
    check = cv2.imread(str(out))
    print(f"{cam}: {src.name} -> {out.relative_to(REPO)}  {w}x{h}  re-read ok={check is not None and check.shape == frame.shape}")

cam_1: cam_1_take3.mp4 -> data/calibration/cam_1_frame.png  1280x720  re-read ok=True


cam_2: cam_2_take3.mp4 -> data/calibration/cam_2_frame.png  1280x720  re-read ok=True
attendance_cam: attendance_cam_take3.mp4 -> data/calibration/attendance_cam_frame.png  1280x720  re-read ok=True


## 2. Calibration points (fill in by hand)

In [3]:
# image_points: pixel coordinates (x, y) of four floor points in the camera image
#               (data/calibration/<cam_id>_frame.png; x to the right, y down, origin top-left).
# map_points:   the same four points on the Room 204 map, normalised to [0, 1]
#               (x to the right, y down). The corridor region is x < 0.3.
# Same order in both lists. Four pairs give an exact fit; more pairs give a least-squares fit.
#
# No camera shows four fixed floor landmarks off one line, so cam_1 / cam_2 pairs are generated
# by the ground model (estimate). attendance_cam sees faces, not feet, so it gets no homography:
# a person it detects is placed at ATTENDANCE_CAM_POINT (just inside the door) instead.
import ground_model

GROUND = {}
CALIB = {"attendance_cam": {"image_points": [], "map_points": []}}
for cam in ("cam_1", "cam_2"):
    GROUND[cam], img_pts, map_pts = ground_model.point_pairs(cam)
    CALIB[cam] = {"image_points": img_pts, "map_points": map_pts}
    g = GROUND[cam]
    print(f"{cam}: horizon {g['horizon_row']:.0f}px, height {g['height_m']:.2f} m ({g['boxes_used']} boxes), "
          f"yaw {g['yaw_deg']:.0f} deg, door {g['door_width_m']:.2f} m, camera at map "
          f"({g['position_m'][0] * ground_model.SX:.3f}, {g['position_m'][1] * ground_model.SY:.3f})")
ATTENDANCE_CAM_POINT = tuple(ground_model.MAP["cameras"]["attendance_cam"][k] for k in ("x", "y"))
CORRIDOR_MAX_X = 0.3

cam_1: horizon 406px, height 0.85 m (225 boxes), yaw 47 deg, door 1.47 m, camera at map (0.066, 0.269)
cam_2: horizon 412px, height 1.13 m (286 boxes), yaw 172 deg, door 1.28 m, camera at map (0.791, 0.485)


## 3. Homography

In [4]:
def compute_homography(cam, points=CALIB):
    """H (3x3) from the calibration points of `cam`, saved to data/calibration/<cam>_homography.json.
    Returns None (and saves nothing) while fewer than four valid pairs are filled in."""
    img = np.asarray(points[cam]["image_points"], dtype=np.float64).reshape(-1, 2)
    mp = np.asarray(points[cam]["map_points"], dtype=np.float64).reshape(-1, 2)
    if len(img) < 4 or len(img) != len(mp):
        print(f"{cam}: skipped, need >= 4 pairs with matching counts (have {len(img)} image, {len(mp)} map)")
        return None
    if not ((mp >= -0.05) & (mp <= 1.05)).all():
        print(f"{cam}: skipped, map_points must be normalised to [0, 1]")
        return None
    H, _ = cv2.findHomography(img, mp, 0)  # 0 = plain least squares over all pairs
    if H is None:
        print(f"{cam}: skipped, points are degenerate (e.g. three on one line)")
        return None
    out = CALIB_DIR / f"{cam}_homography.json"
    out.write_text(json.dumps({"cam_id": cam, "take": TAKE, "frame_size": FRAME_SIZE.get(cam),
                               "image_points": img.tolist(), "map_points": mp.tolist(),
                               "H": H.tolist(),
                               "method": "ground_model.py estimate (no floor markers)" if cam in GROUND else "measured points"},
                              indent=1), encoding="utf-8")
    print(f"{cam}: H saved to {out.relative_to(REPO)}")
    return H

H = {cam: compute_homography(cam) for cam in CAMS}

cam_1: H saved to data/calibration/cam_1_homography.json
cam_2: H saved to data/calibration/cam_2_homography.json
attendance_cam: skipped, need >= 4 pairs with matching counts (have 0 image, 0 map)


## 4. Box → map

Track boxes follow the data contract: `[x, y, w, h]` as fractions of the frame, `x, y` = **top-left**.
The feet are the bottom-centre: `((x + w/2) * width, (y + h) * height)` in pixels, the same space as `image_points`.

In [5]:
def foot_to_map(box, H, frame_size=(1280, 720)):
    """Map a normalised top-left [x, y, w, h] box to (map_x, map_y) on the Room 204 map.
    frame_size = (width, height) in pixels of the image the points were picked on (all cameras record 1280x720)."""
    if H is None:
        raise ValueError("no homography for this camera yet: fill in its calibration points")
    x, y, w, h = box
    fw, fh = frame_size
    foot = np.array([[[(x + w / 2) * fw, (y + h) * fh]]], dtype=np.float64)
    mx, my = cv2.perspectiveTransform(foot, np.asarray(H, dtype=np.float64))[0, 0]
    return float(mx), float(my)

def in_corridor(map_xy):
    return map_xy[0] < CORRIDOR_MAX_X

def box_to_map(cam, box, H_by_cam, frame_size=(1280, 720)):
    """Map position of a tracked box, or None when the feet are not usable.
    attendance_cam -> fixed point at the door. A box touching the bottom edge has its feet cut off;
    a box whose bottom is high in the frame is usually a seated person behind a table (cam_2) or
    someone seen through the glass (cam_1), so the projection is not trusted either."""
    if cam == "attendance_cam":
        return ATTENDANCE_CAM_POINT
    x, y, w, h = box
    if y + h >= 0.995 or H_by_cam.get(cam) is None:
        return None
    return foot_to_map(box, H_by_cam[cam], frame_size)

## 5. Checks (synthetic points only — not calibration data)

In [6]:
# Calibration state: cam_1 / cam_2 solved from the ground model, attendance_cam uses the door point.
assert H["cam_1"] is not None and H["cam_2"] is not None and H["attendance_cam"] is None
assert all((CALIB_DIR / f"{c}_homography.json").exists() for c in ("cam_1", "cam_2"))
assert box_to_map("attendance_cam", [0.4, 0.2, 0.2, 0.6], H) == ATTENDANCE_CAM_POINT
assert box_to_map("cam_1", [0.4, 0.5, 0.2, 0.5], H) is None   # feet cut off by the frame edge
try:
    foot_to_map([0.4, 0.5, 0.2, 0.3], None)
except ValueError as e:
    print("missing H handled:", e)
print("frames saved:", sorted(p.name for p in CALIB_DIR.glob("*_frame.png")))

# Synthetic 1: image corners -> unit square. The feet of [0.4, 0.5, 0.2, 0.3] are at (0.5, 0.8).
W, Hh = 1280, 720
syn_img = [[0, 0], [W, 0], [W, Hh], [0, Hh]]
syn_map = [[0, 0], [1, 0], [1, 1], [0, 1]]
Hs, _ = cv2.findHomography(np.float64(syn_img), np.float64(syn_map), 0)
p = foot_to_map([0.4, 0.5, 0.2, 0.3], Hs, (W, Hh))
assert np.allclose(p, (0.5, 0.8)), p
print("synthetic scale:", np.round(p, 6))

# Synthetic 2: a perspective trapezoid -> unit square maps its corners exactly.
trap = [[400, 300], [880, 300], [1280, 720], [0, 720]]
Ht, _ = cv2.findHomography(np.float64(trap), np.float64(syn_map), 0)
back = cv2.perspectiveTransform(np.float64(trap).reshape(-1, 1, 2), Ht).reshape(-1, 2)
assert np.allclose(back, syn_map, atol=1e-9), back
print("synthetic perspective: corners exact")

# Synthetic 3: compute_homography() rejects bad input without crashing.
for bad in ({"image_points": syn_img[:3], "map_points": syn_map[:3]},
            {"image_points": syn_img, "map_points": [[0, 0], [2, 0], [2, 2], [0, 2]]}):
    assert compute_homography("cam_X", {"cam_X": bad}) is None
assert not (CALIB_DIR / "cam_X_homography.json").exists()
print("all checks passed")

missing H handled: no homography for this camera yet: fill in its calibration points
frames saved: ['attendance_cam_frame.png', 'cam_1_frame.png', 'cam_2_frame.png']
synthetic scale: [0.5 0.8]
synthetic perspective: corners exact
cam_X: skipped, need >= 4 pairs with matching counts (have 3 image, 3 map)
cam_X: skipped, map_points must be normalised to [0, 1]
all checks passed


## 6. Sanity check on the demo-take tracks (take 3)

Ground truth (take3.csv): the three students walk down the corridor (cam_1, ~8–11 s), stand in the
doorway (~13–21 s) and then sit in the room (cam_2 after ~24 s). Only boxes with usable feet are mapped.

In [7]:
tracks = {c: json.loads((REPO / "data" / "tracks" / f"take{TAKE}" / f"{c}.json").read_text())["tracks"] for c in CAMS}
# take-3 track ids to leave out: P4 seated the whole video (cam_1 1, cam_2 1) and the poster of a man
# on the classroom wall that YOLO detects as a person through the glass (cam_1 3).
SEATED_OR_GLASS = {"cam_1": {1, 3}, "cam_2": {1}}

def mapped(cam, t0, t1, min_bottom=0.0):
    out = []
    for tr in tracks[cam]:
        if tr["global_id"] in SEATED_OR_GLASS.get(cam, ()):
            continue
        for k in tr["keyframes"]:
            if t0 <= k["t"] <= t1 and k["box"][1] + k["box"][3] >= min_bottom:
                p = box_to_map(cam, k["box"], H)
                if p is not None:
                    out.append(p)
    return np.array(out)

cases = [("cam_1", "corridor walk 8.5-11 s -> x < 0.3", 8.5, 11, 0.0, lambda p: p[:, 0] < CORRIDOR_MAX_X),
         ("cam_1", "doorway 13-21 s -> near the door", 13, 21, 0.0,
          lambda p: (abs(p[:, 0] - 0.3) < 0.06) & (p[:, 1] > 0.72) & (p[:, 1] < 1.02)),
         ("cam_2", "inside the door 11-21 s -> near the door", 11, 21, 0.8,
          lambda p: (p[:, 0] > 0.28) & (p[:, 0] < 0.45) & (p[:, 1] > 0.6) & (p[:, 1] < 1.02)),
         ("cam_2", "after 24 s, feet on visible floor -> in the room", 24, 31.4, 0.8,
          lambda p: (p[:, 0] >= 0.28) & (p[:, 0] <= 1) & (p[:, 1] >= 0) & (p[:, 1] <= 1.02))]
for cam, name, t0, t1, mb, ok in cases:
    p = mapped(cam, t0, t1, mb)
    rate = ok(p).mean()
    print(f"{cam} {name}: {ok(p).sum()}/{len(p)} = {rate:.0%}; median ({np.median(p[:, 0]):.2f}, {np.median(p[:, 1]):.2f})")
    assert rate >= 0.9, "calibration sanity check failed"

cam_1 corridor walk 8.5-11 s -> x < 0.3: 48/53 = 91%; median (0.22, 0.56)
cam_1 doorway 13-21 s -> near the door: 97/97 = 100%; median (0.29, 0.79)
cam_2 inside the door 11-21 s -> near the door: 243/248 = 98%; median (0.34, 0.79)
cam_2 after 24 s, feet on visible floor -> in the room: 135/138 = 98%; median (0.43, 0.43)
